# BanglaBERT + CNN-BiLSTM Hybrid — Sentiment Analysis (Kaggle) — v3

Fine-tuned `csebuetnlp/banglabert` fused with a Conv1D->BiLSTM branch, class-weighted +
label-smoothed loss, discriminative-LR AdamW, partial encoder freezing, early stopping on
validation macro-F1. Runs both the 3-class and 2-class tasks on the original ICCIT 2020
paper's dataset and compares against its reported **60% (3-class)** / **71% (2-class)** accuracy.

**v2 changes**, in response to round-1 overfitting (train loss 0.91->0.20 while val loss rose
0.87->2.11 on 3-class; best-val checkpoint scored 0.65 val macro-F1 but only 0.50 test
macro-F1): lower head/encoder LR, label smoothing, higher dropout, partial encoder-layer
freezing, an LR-decay horizon decoupled from the early-stopping epoch cap, and per-epoch
train/val plots so the overfitting curve is visible directly instead of inferred from logs.

**v3 changes (Phase 2 -- maximize training signal)**: rounds 1-2 both showed val macro-F1
peaking in the epoch 2-4 range then declining, with a 15% validation split. To give the final
run more training signal while still selecting the epoch dynamically (not by a number fixed in
advance, and never by peeking at the test set), `val_ratio` is shrunk to the minimum needed for
a stable early-stopping signal (5%) rather than removed entirely -- early stopping has no signal
to act on without *some* held-out split. The test set remains fully held out until the single
final evaluation per task.

Sources code from `https://github.com/habibour/sent` and reads
`/kaggle/input/datasets/reversedthoutgts/bangla-dataset/{train_,test_}.csv`.

To smoke-test before committing to a full run, lower `EPOCHS` in the config cell below to 1-2.


In [ ]:
import os

REPO_DIR = '/kaggle/working/sent'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/habibour/sent.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(REPO_DIR)


In [ ]:
!pip install -q -U transformers scikit-learn
!pip install -q git+https://github.com/csebuetnlp/normalizer.git


In [ ]:
import random
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.data_utils import load_and_prepare, get_class_weights, verify_normalizer
from src.hybrid_model import (
    SentimentDataset, BanglaBertHybrid, freeze_encoder_layers,
    train_with_early_stopping, evaluate, plot_history, MAX_LEN,
)
from src.hierarchical import (
    load_and_prepare_hierarchical, predict_hierarchical, per_class_recall,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

verify_normalizer()


## Config

`PAPER_BASELINE` is each task's accuracy from Table V of the original paper (BERT_BSA+GRU).

Round-1 defaults were `encoder_lr=2e-5, head_lr=1e-3, dropout=0.3, num_frozen_layers=0`, which
overfit. v2 defaults below are more conservative; adjust further if the plots still show the
val loss diverging early.

`val_ratio` is 0.05 (down from 0.15 in v1/v2) as of v3: rounds 1-2 already established that val
macro-F1 peaks early (epoch 2-4) and declines after, so the val split's job now is only to give
early stopping a stable enough signal to find that peak -- not to support further
hyperparameter tuning -- so it's kept as small as that job allows, maximizing training data.


In [ ]:
CONFIG = {
    'train_path': '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/train_.csv',
    'test_path': '/kaggle/input/datasets/reversedthoutgts/bangla-dataset/test_.csv',
    'model_name': 'csebuetnlp/banglabert',
    'max_len': MAX_LEN,
    'val_ratio': 0.05,      # was 0.15 -- v3: shrink to the minimum early stopping needs
    'seed': 42,
    'batch_size': 16,
    'epochs': 15,           # hard cap; early stopping usually ends it sooner
    'lr_decay_epochs': 8,   # horizon the LR schedule decays over
    'patience': 3,
    'encoder_lr': 1e-5,     # was 2e-5
    'head_lr': 3e-4,        # was 1e-3
    'dropout': 0.4,         # was 0.3
    'label_smoothing': 0.1, # new
    'num_frozen_layers': 6, # new -- freeze bottom half of BanglaBERT's 12 layers
    'weight_decay': 0.01,
    'warmup_ratio': 0.06,
    'grad_clip': 1.0,
    'use_fp16': True,
    'output_dir': '/kaggle/working',
}

TASKS = {
    '3class': {'num_labels': 3, 'label_names': ['Neutral', 'Positive', 'Negative'], 'paper_baseline': 0.60},
    '2class': {'num_labels': 2, 'label_names': ['Negative', 'Positive'], 'paper_baseline': 0.71},
}

print(CONFIG)
print(TASKS)


## Experiment function

Runs the full pipeline for one task: data prep -> tokenize -> fine-tune -> plot -> evaluate on held-out test.

In [ ]:
def run_experiment(task, weight_multiplier=None):
    """weight_multiplier, if given, is a {label: factor} dict applied on top
    of balanced class weights (see get_class_weights) -- used by the Phase 3
    weighted final run, after the sweep below has picked a factor using val
    metrics only."""
    assert task in TASKS, f"task must be one of {list(TASKS)}"
    cfg = TASKS[task]
    print('='*30, task, '='*30)
    set_seed(CONFIG['seed'])

    train_df, val_df, test_df = load_and_prepare(
        CONFIG['train_path'], CONFIG['test_path'], task,
        val_ratio=CONFIG['val_ratio'], seed=CONFIG['seed'],
    )

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

    train_ds = SentimentDataset(train_df['Data'], train_df['label'], tokenizer, CONFIG['max_len'])
    val_ds = SentimentDataset(val_df['Data'], val_df['label'], tokenizer, CONFIG['max_len'])
    test_ds = SentimentDataset(test_df['Data'], test_df['label'], tokenizer, CONFIG['max_len'])

    train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    model = BanglaBertHybrid(CONFIG['model_name'], num_labels=cfg['num_labels'],
                              dropout=CONFIG['dropout']).to(device)

    if CONFIG['num_frozen_layers'] > 0:
        freeze_encoder_layers(model, CONFIG['num_frozen_layers'])

    class_weights = get_class_weights(train_df['label'].values, cfg['num_labels'],
                                       multipliers=weight_multiplier).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['label_smoothing'])
    print('class weights:', class_weights)

    start = time.time()
    model, best_val_macro_f1, history = train_with_early_stopping(
        model, train_loader, val_loader, criterion, device,
        encoder_lr=CONFIG['encoder_lr'], head_lr=CONFIG['head_lr'],
        weight_decay=CONFIG['weight_decay'], warmup_ratio=CONFIG['warmup_ratio'],
        epochs=CONFIG['epochs'], lr_decay_epochs=CONFIG['lr_decay_epochs'],
        patience=CONFIG['patience'], grad_clip=CONFIG['grad_clip'],
        use_fp16=CONFIG['use_fp16'], label_names=cfg['label_names'],
    )
    elapsed = time.time() - start
    print(f'training time: {elapsed/60:.1f} min, best val macro-F1: {best_val_macro_f1:.4f}')

    plot_history(history, title=task)

    ckpt_path = f"{CONFIG['output_dir']}/banglabert_hybrid_{task}.pt"
    torch.save(model.state_dict(), ckpt_path)

    test_metrics = evaluate(model, test_loader, criterion, device, label_names=cfg['label_names'])
    print(f"\nTest accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Test macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"Test weighted-F1: {test_metrics['weighted_f1']:.4f}")
    print('\n' + test_metrics['report'])
    print('Confusion matrix:\n', test_metrics['confusion_matrix'])

    paper_acc = cfg['paper_baseline']
    print(f"\nPaper's reported BERT_BSA+GRU {task} accuracy: {paper_acc*100:.2f}%")
    print(f"This run's test accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Difference: {(test_metrics['accuracy'] - paper_acc)*100:+.2f} percentage points")

    return {
        'task': task,
        'accuracy': test_metrics['accuracy'],
        'macro_f1': test_metrics['macro_f1'],
        'weighted_f1': test_metrics['weighted_f1'],
        'paper_baseline': paper_acc,
        'best_val_macro_f1': best_val_macro_f1,
        'checkpoint': ckpt_path,
        'history': history,
        'confusion_matrix': test_metrics['confusion_matrix'],
    }


## Run both tasks

In [ ]:
results = []
for task in ['3class', '2class']:
    results.append(run_experiment(task))


In [ ]:
summary = pd.DataFrame([{k: v for k, v in r.items() if k not in ('history', 'confusion_matrix')}
                         for r in results])
summary['improvement_pp'] = (summary['accuracy'] - summary['paper_baseline']) * 100
print(summary.to_string(index=False))
summary.to_csv(f"{CONFIG['output_dir']}/results_summary.csv", index=False)


## Phase 3 — minority-class targeted fixes

The flat 3-class confusion matrix from Phase 1/2 showed the actual bottleneck: Neutral
(the minority class, and semantically the boundary between the other two) getting absorbed
into Negative. This phase targets that directly, in two independent ways, then compares them:

1. **Class-weight sweep**: push the under-recalled class's weight beyond plain
   inverse-frequency balancing (Neutral for 3class, Positive for 2class), watching for
   majority-class recall collapse as the cost. Selected on **validation** metrics only --
   the test set is not touched during the sweep, same discipline as the Phase 2 epoch
   decision, so picking a multiplier doesn't quietly become tuning against the test set.
2. **Hierarchical two-stage classifier** (3-class only): stage 1 decides Neutral-vs-rest,
   stage 2 decides Positive-vs-Negative only on what stage 1 passes through. This gives the
   Neutral-vs-rest confusion its own dedicated binary decision instead of a shared 3-way head.
3. **Head-to-head comparison** of the flat vs. hierarchical classifier on the same test set
   and training recipe, including per-class recall so the Neutral-recall gain (if any) and
   its cost to Positive/Negative recall are both visible.


In [ ]:
def run_weight_sweep(task, target_label, multipliers):
    """Train BanglaBertHybrid once per multiplier in `multipliers`, each time
    scaling class_weights[target_label] by that factor on top of balanced
    weighting (see get_class_weights), same data/recipe as run_experiment
    otherwise. Reports **validation** metrics and per-class validation recall
    only -- the test set is never touched here, so the sweep can't turn into
    tuning against the test set. Returns a DataFrame, one row per multiplier.
    """
    cfg = TASKS[task]
    rows = []
    for m in multipliers:
        print('='*20, task, f'weight[{target_label}] x{m}', '='*20)
        set_seed(CONFIG['seed'])

        train_df, val_df, test_df = load_and_prepare(
            CONFIG['train_path'], CONFIG['test_path'], task,
            val_ratio=CONFIG['val_ratio'], seed=CONFIG['seed'],
        )
        tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
        train_ds = SentimentDataset(train_df['Data'], train_df['label'], tokenizer, CONFIG['max_len'])
        val_ds = SentimentDataset(val_df['Data'], val_df['label'], tokenizer, CONFIG['max_len'])
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

        model = BanglaBertHybrid(CONFIG['model_name'], num_labels=cfg['num_labels'],
                                  dropout=CONFIG['dropout']).to(device)
        if CONFIG['num_frozen_layers'] > 0:
            freeze_encoder_layers(model, CONFIG['num_frozen_layers'])

        class_weights = get_class_weights(train_df['label'].values, cfg['num_labels'],
                                           multipliers={target_label: m}).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['label_smoothing'])
        print('class weights:', class_weights)

        model, best_val_macro_f1, history = train_with_early_stopping(
            model, train_loader, val_loader, criterion, device,
            encoder_lr=CONFIG['encoder_lr'], head_lr=CONFIG['head_lr'],
            weight_decay=CONFIG['weight_decay'], warmup_ratio=CONFIG['warmup_ratio'],
            epochs=CONFIG['epochs'], lr_decay_epochs=CONFIG['lr_decay_epochs'],
            patience=CONFIG['patience'], grad_clip=CONFIG['grad_clip'],
            use_fp16=CONFIG['use_fp16'], label_names=cfg['label_names'],
        )
        val_metrics = evaluate(model, val_loader, criterion, device, label_names=cfg['label_names'])
        recalls = per_class_recall(val_metrics['confusion_matrix'])

        row = {
            'multiplier': m,
            'val_macro_f1': best_val_macro_f1,
            'val_accuracy': val_metrics['accuracy'],
        }
        for i, name in enumerate(cfg['label_names']):
            row[f'val_recall_{name}'] = recalls[i]
        print(row)
        rows.append(row)

    return pd.DataFrame(rows)


### Run the sweep

3-class: push Neutral's weight (label `0`). 2-class: push Positive's weight (label `1`,
per `to_2class_labels`'s `{Negative -> 0, Positive -> 1}` mapping). `1.0` is the plain
balanced-weight baseline, included for direct comparison. Shrink this list for a smoke test.


In [ ]:
sweep_3class = run_weight_sweep('3class', target_label=0, multipliers=[1.0, 1.5, 2.0, 3.0])
print('\n3-class Neutral-weight sweep (validation only):')
print(sweep_3class.to_string(index=False))


In [ ]:
sweep_2class = run_weight_sweep('2class', target_label=1, multipliers=[1.0, 1.5, 2.0, 3.0])
print('\n2-class Positive-weight sweep (validation only):')
print(sweep_2class.to_string(index=False))


### Pick a multiplier

Inspect the two tables above and pick the multiplier for each task that improves the
target class's recall without collapsing the other classes' recall too far -- there's no
single automatic rule here since "too far" is a judgment call specific to what this model is
for; set it below once decided. `None` means "keep plain balanced weighting" (no Phase 3
reweighting applied) for that task.


In [ ]:
CHOSEN_MULTIPLIER = {
    '3class': None,  # e.g. {0: 2.0} to push Neutral 2x -- fill in from sweep_3class above
    '2class': None,  # e.g. {1: 2.0} to push Positive 2x -- fill in from sweep_2class above
}


### Final weighted run (test set touched once)

Retrains each task with the chosen multiplier (or plain balanced weighting if left `None`)
and evaluates on the held-out test set exactly once per task -- this is the only place in
Phase 3's weight-sweep track where the test set is used.


In [ ]:
weighted_results = {
    task: run_experiment(task, weight_multiplier=CHOSEN_MULTIPLIER[task])
    for task in ['3class', '2class']
}


## Hierarchical two-stage classifier (3-class)

Trains two independent binary `BanglaBertHybrid` models -- stage 1 (Neutral-vs-rest) and
stage 2 (Positive-vs-Negative, trained only on non-Neutral rows) -- then cascades them at
inference time and scores the combined predictions against the same 3-class test labels
the flat classifier is scored on. Same training recipe (`CONFIG`) as the flat run, so the
comparison below isolates the architecture change.


In [ ]:
def run_hierarchical_experiment():
    print('='*30, 'hierarchical (3class)', '='*30)
    set_seed(CONFIG['seed'])

    stage1_train, stage1_val, stage2_train, stage2_val, test_df = load_and_prepare_hierarchical(
        CONFIG['train_path'], CONFIG['test_path'],
        val_ratio=CONFIG['val_ratio'], seed=CONFIG['seed'],
    )
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

    def make_loaders(train_df, val_df):
        train_ds = SentimentDataset(train_df['Data'], train_df['label'], tokenizer, CONFIG['max_len'])
        val_ds = SentimentDataset(val_df['Data'], val_df['label'], tokenizer, CONFIG['max_len'])
        train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
        val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)
        return train_loader, val_loader

    stage_data = {
        'stage1': (stage1_train, stage1_val, ['Rest', 'Neutral']),
        'stage2': (stage2_train, stage2_val, ['Negative', 'Positive']),
    }
    stage_models, stage_val_f1 = {}, {}
    for stage_name, (train_df, val_df, label_names) in stage_data.items():
        print('-'*20, stage_name, '-'*20)
        train_loader, val_loader = make_loaders(train_df, val_df)

        model = BanglaBertHybrid(CONFIG['model_name'], num_labels=2,
                                  dropout=CONFIG['dropout']).to(device)
        if CONFIG['num_frozen_layers'] > 0:
            freeze_encoder_layers(model, CONFIG['num_frozen_layers'])

        class_weights = get_class_weights(train_df['label'].values, 2).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['label_smoothing'])

        model, best_val_macro_f1, history = train_with_early_stopping(
            model, train_loader, val_loader, criterion, device,
            encoder_lr=CONFIG['encoder_lr'], head_lr=CONFIG['head_lr'],
            weight_decay=CONFIG['weight_decay'], warmup_ratio=CONFIG['warmup_ratio'],
            epochs=CONFIG['epochs'], lr_decay_epochs=CONFIG['lr_decay_epochs'],
            patience=CONFIG['patience'], grad_clip=CONFIG['grad_clip'],
            use_fp16=CONFIG['use_fp16'], label_names=label_names,
        )
        plot_history(history, title=f'hierarchical-{stage_name}')
        print(f'{stage_name} best val macro-F1: {best_val_macro_f1:.4f}')

        ckpt_path = f"{CONFIG['output_dir']}/banglabert_hierarchical_{stage_name}.pt"
        torch.save(model.state_dict(), ckpt_path)
        stage_models[stage_name] = model
        stage_val_f1[stage_name] = best_val_macro_f1

    test_ds = SentimentDataset(test_df['Data'], test_df['label'], tokenizer, CONFIG['max_len'])
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    test_metrics = predict_hierarchical(stage_models['stage1'], stage_models['stage2'],
                                         test_loader, device)
    print(f"\nHierarchical test accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Hierarchical test macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"Hierarchical test weighted-F1: {test_metrics['weighted_f1']:.4f}")
    print('\n' + test_metrics['report'])
    print('Confusion matrix:\n', test_metrics['confusion_matrix'])

    return {
        'task': '3class-hierarchical',
        'accuracy': test_metrics['accuracy'],
        'macro_f1': test_metrics['macro_f1'],
        'weighted_f1': test_metrics['weighted_f1'],
        'stage1_val_macro_f1': stage_val_f1['stage1'],
        'stage2_val_macro_f1': stage_val_f1['stage2'],
        'confusion_matrix': test_metrics['confusion_matrix'],
    }

hierarchical_result = run_hierarchical_experiment()


## Flat vs. hierarchical — head-to-head

Same test set, same underlying training recipe. `flat_3class_result` here is the plain
balanced-weight run from the "Run both tasks" section above (not the Phase 3 reweighted one),
so this isolates the architecture change specifically; compare against `weighted_results['3class']`
separately if you want the reweighted flat model instead.


In [ ]:
flat_3class_result = next(r for r in results if r['task'] == '3class')

comparison = pd.DataFrame([
    {'model': 'flat', 'accuracy': flat_3class_result['accuracy'],
     'macro_f1': flat_3class_result['macro_f1'], 'weighted_f1': flat_3class_result['weighted_f1']},
    {'model': 'hierarchical', 'accuracy': hierarchical_result['accuracy'],
     'macro_f1': hierarchical_result['macro_f1'], 'weighted_f1': hierarchical_result['weighted_f1']},
])
print(comparison.to_string(index=False))

recall_comparison = pd.DataFrame({
    'class': TASKS['3class']['label_names'],
    'flat_recall': per_class_recall(flat_3class_result['confusion_matrix']),
    'hierarchical_recall': per_class_recall(hierarchical_result['confusion_matrix']),
})
print('\nPer-class recall (the Neutral row is the one that matters most here):')
print(recall_comparison.to_string(index=False))

comparison.to_csv(f"{CONFIG['output_dir']}/flat_vs_hierarchical.csv", index=False)
recall_comparison.to_csv(f"{CONFIG['output_dir']}/flat_vs_hierarchical_recall.csv", index=False)


## Phase 2 — full-data training (fixed epoch schedule, no held-out validation)

The val split above (`train_with_early_stopping`) exists to answer one question: *which epoch
should training stop at?* Once that question is answered from the runs above, holding those
val rows out of the final model forever is pure waste -- they were never going to be looked at
again, and folding them back into training gives the model ~5-15% more labeled examples for the
same held-out `test.csv` comparison (methodology stays identical: only train/val composition
changes, the test set is untouched).

`FIXED_EPOCHS` below is read off where val-macro-F1 peaked in the round-1/round-2 runs already
completed (see run history / plots above): 3-class peaked at epoch 3-4, 2-class at epoch 2-4
(tied). Picking the slightly higher end of each range costs little (this schedule has no early
stopping to overshoot into overfitting territory across such a small epoch gap) and errs toward
using the extra data fully. If you re-run the val-based experiment above after further changes,
update `FIXED_EPOCHS` from the new history before trusting this section again -- it is not
re-derived automatically.


In [ ]:
FIXED_EPOCHS = {
    '3class': 4,  # val-macro-F1 peaked at epoch 3 (round 1) and epoch 4 (round 2)
    '2class': 3,  # val-macro-F1 peaked at epoch 2 (round 1); epochs 2-4 ~tied (round 2)
}


In [ ]:
def run_full_data_experiment(task, fixed_epochs, weight_multiplier=None):
    """Phase 2: retrain on train+val combined (no held-out val, no early
    stopping) for exactly `fixed_epochs` epochs, then evaluate on the same
    held-out test.csv used everywhere else. `fixed_epochs` must come from a
    prior train_with_early_stopping run's history (see FIXED_EPOCHS above) --
    this function does not pick its own epoch count.

    `weight_multiplier` behaves the same as in run_experiment/run_weight_sweep
    (a {label: factor} dict on top of balanced class weights), so this can
    also be used to combine Phase 2 (full data) with Phase 3 (minority
    reweighting) by passing CHOSEN_MULTIPLIER[task].
    """
    from src.hybrid_model import train_fixed_epochs

    assert task in TASKS, f"task must be one of {list(TASKS)}"
    cfg = TASKS[task]
    print('='*30, f'{task} (full-data, fixed {fixed_epochs} epochs)', '='*30)
    set_seed(CONFIG['seed'])

    train_df, val_df, test_df = load_and_prepare(
        CONFIG['train_path'], CONFIG['test_path'], task,
        val_ratio=CONFIG['val_ratio'], seed=CONFIG['seed'],
    )
    full_train_df = pd.concat([train_df, val_df], ignore_index=True)
    print(f"[full-data] combined train+val: {len(full_train_df)} rows "
          f"(train {len(train_df)} + val {len(val_df)}); test untouched: {len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

    full_train_ds = SentimentDataset(full_train_df['Data'], full_train_df['label'], tokenizer, CONFIG['max_len'])
    test_ds = SentimentDataset(test_df['Data'], test_df['label'], tokenizer, CONFIG['max_len'])

    full_train_loader = DataLoader(full_train_ds, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=2)

    model = BanglaBertHybrid(CONFIG['model_name'], num_labels=cfg['num_labels'],
                              dropout=CONFIG['dropout']).to(device)

    if CONFIG['num_frozen_layers'] > 0:
        freeze_encoder_layers(model, CONFIG['num_frozen_layers'])

    class_weights = get_class_weights(full_train_df['label'].values, cfg['num_labels'],
                                       multipliers=weight_multiplier).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=CONFIG['label_smoothing'])
    print('class weights:', class_weights)

    start = time.time()
    model, history = train_fixed_epochs(
        model, full_train_loader, criterion, device,
        encoder_lr=CONFIG['encoder_lr'], head_lr=CONFIG['head_lr'],
        weight_decay=CONFIG['weight_decay'], warmup_ratio=CONFIG['warmup_ratio'],
        epochs=fixed_epochs, grad_clip=CONFIG['grad_clip'], use_fp16=CONFIG['use_fp16'],
    )
    elapsed = time.time() - start
    print(f'training time: {elapsed/60:.1f} min')

    ckpt_path = f"{CONFIG['output_dir']}/banglabert_hybrid_{task}_fulldata.pt"
    torch.save(model.state_dict(), ckpt_path)

    test_metrics = evaluate(model, test_loader, criterion, device, label_names=cfg['label_names'])
    print(f"\nTest accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Test macro-F1: {test_metrics['macro_f1']:.4f}")
    print(f"Test weighted-F1: {test_metrics['weighted_f1']:.4f}")
    print('\n' + test_metrics['report'])
    print('Confusion matrix:\n', test_metrics['confusion_matrix'])

    paper_acc = cfg['paper_baseline']
    print(f"\nPaper's reported BERT_BSA+GRU {task} accuracy: {paper_acc*100:.2f}%")
    print(f"This run's test accuracy: {test_metrics['accuracy']*100:.2f}%")
    print(f"Difference: {(test_metrics['accuracy'] - paper_acc)*100:+.2f} percentage points")

    return {
        'task': task,
        'accuracy': test_metrics['accuracy'],
        'macro_f1': test_metrics['macro_f1'],
        'weighted_f1': test_metrics['weighted_f1'],
        'paper_baseline': paper_acc,
        'fixed_epochs': fixed_epochs,
        'checkpoint': ckpt_path,
        'history': history,
        'confusion_matrix': test_metrics['confusion_matrix'],
    }


### Run full-data experiments

Plain balanced weighting (no `weight_multiplier`) here, to isolate the effect of Phase 2
(more training data, fixed epochs) on its own -- comparable to the plain `results` from the
"Run both tasks" section, not the Phase 3 reweighted run. Combining Phase 2 + Phase 3 (full
data *and* minority reweighting) is one line away if wanted: pass
`weight_multiplier=CHOSEN_MULTIPLIER[task]` below once that dict is filled in.


In [ ]:
full_data_results = {
    task: run_full_data_experiment(task, FIXED_EPOCHS[task])
    for task in ['3class', '2class']
}


## Consolidated comparison -- flat vs. weighted vs. hierarchical vs. full-data

Same test set throughout; each row differs only in training methodology. `flat` is the plain
balanced-weight run from "Run both tasks"; `weighted` is Phase 3's targeted-multiplier run;
`hierarchical` is the two-stage cascade (3-class only); `full-data` is this Phase 2 section.


In [ ]:
final_rows = []
for r in results:
    final_rows.append({'task': r['task'], 'method': 'flat', 'accuracy': r['accuracy'],
                        'macro_f1': r['macro_f1'], 'weighted_f1': r['weighted_f1']})
for task, r in weighted_results.items():
    final_rows.append({'task': task, 'method': 'weighted', 'accuracy': r['accuracy'],
                        'macro_f1': r['macro_f1'], 'weighted_f1': r['weighted_f1']})
final_rows.append({'task': '3class', 'method': 'hierarchical',
                    'accuracy': hierarchical_result['accuracy'],
                    'macro_f1': hierarchical_result['macro_f1'],
                    'weighted_f1': hierarchical_result['weighted_f1']})
for task, r in full_data_results.items():
    final_rows.append({'task': task, 'method': 'full-data', 'accuracy': r['accuracy'],
                        'macro_f1': r['macro_f1'], 'weighted_f1': r['weighted_f1']})

final_summary = pd.DataFrame(final_rows)
print(final_summary.to_string(index=False))
final_summary.to_csv(f"{CONFIG['output_dir']}/final_comparison_summary.csv", index=False)
